# Fine-tune `kharcha-extract` (Qwen2.5-0.5B)

Runs on **Google Colab (T4)** or a local NVIDIA GPU. Produces a Q4 GGUF you import into Ollama.

## Steps
1. Upload `data/extraction/finetune_chat.jsonl` (from `python -m scripts.prepare_llm_finetune`).
2. Runtime → Change runtime type → GPU.
3. Run all cells.
4. Download `kharcha-extract-q4_k_m.gguf` into `data/extraction/`.
5. `ollama create kharcha-extract -f data/extraction/Modelfile`
6. Set `OLLAMA_MODEL=kharcha-extract` in `.env`.

Goal: smaller/faster residual LLM path with valid JSON on the first try (fewer retries).

In [ ]:
# Install Unsloth + export deps (Colab)
%pip install -q --upgrade pip
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps "xformers" "trl" "peft" "accelerate" "bitsandbytes"
%pip install -q datasets transformers

In [ ]:
from pathlib import Path

# Colab: upload finetune_chat.jsonl via Files panel, or set path after drive mount.
DATA_PATH = Path("finetune_chat.jsonl")
assert DATA_PATH.is_file(), (
    "Upload data/extraction/finetune_chat.jsonl to the runtime cwd first."
)
print("rows:", sum(1 for _ in DATA_PATH.open()))

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
dtype = None  # auto
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")

def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        texts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        )
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:400])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_steps=20,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="kharcha-extract-checkpoints",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
# Quick smoke: force a residual-style transcript (no digit amount)
FastLanguageModel.for_inference(model)
messages = [
    {
        "role": "system",
        "content": (
            "Extract one expense from a short Hinglish transcript. "
            "Reply with one JSON object only."
        ),
    },
    {
        "role": "user",
        "content": (
            "Transcript: 'chai piyi Rahul ke saath'\n"
            "Hints: amount_paise=None, split=True, category=Food\n"
            "Friends: ['Rahul', 'Priya']\n"
            "Categories: ['Food', 'Transport', 'Rent', 'Shopping', 'Entertainment', 'Other']\n"
            "JSON keys: amount_rupees, description, category_name, friend_names, split, spent_on"
        ),
    },
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=128, temperature=0.0)
print(tokenizer.decode(out[0], skip_special_tokens=True))

In [ ]:
# Export merged 16-bit then GGUF Q4 for Ollama
model.save_pretrained_merged("kharcha-extract-merged", tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf(
    "kharcha-extract-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

from pathlib import Path
ggufs = list(Path("kharcha-extract-gguf").rglob("*.gguf"))
print("GGUF files:", ggufs)
if ggufs:
    target = Path("kharcha-extract-q4_k_m.gguf")
    target.write_bytes(ggufs[0].read_bytes())
    print("Wrote", target.resolve(), "— download this into data/extraction/")

## Local Ollama import

```bash
# from repo root, after placing the GGUF beside the Modelfile:
ollama create kharcha-extract -f data/extraction/Modelfile

# point the API at it
# OLLAMA_MODEL=kharcha-extract

# compare residual latency / JSON-valid@1
python -m scripts.eval_extraction --no-prepass --llm --llm-limit 20
```